In [1]:
import sys
from pathlib import Path


def _find_root(start: Path) -> Path:
    """Walk up from `start` until we find the repo root (has pyproject.toml + quantdata/)."""
    for d in (start, *start.parents):
        if (d / "pyproject.toml").exists() and (d / "quantdata").is_dir():
            return d
    raise RuntimeError(f"quant_data project root not found above {start}")


PROJECT_ROOT = _find_root(Path.cwd().resolve())

# `quantdata` is an editable-installed package, so `import quantdata...` works
# already; PROJECT_ROOT is added so it also imports without the editable install.
# The `personal/` modules are flat scripts (no __init__.py), so every directory
# we import a module from must be on sys.path explicitly.
for p in (
    PROJECT_ROOT,
    PROJECT_ROOT / "personal" / "shared_functions",
    PROJECT_ROOT / "personal" / "SANOS_KF_Strike",
    PROJECT_ROOT / "personal" / "Kalman_filter_spline",
):
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

from datetime import date

import numpy as np
import pandas as pd
import polars as pl

import personal.SANOS_KF_Strike.sanos_ekf_eval as ekf

# personal/ flat modules:
import personal.shared_functions.unified_metrics as um

# quantdata package (example — import whatever you need):
import quantdata  # e.g. from quantdata.sanos import open as sanos_open
from personal.Kalman_filter_spline.surface_kalman_filters import build_iv_observations
from personal.SANOS_KF_Strike.sanos_kf_strike import (
    SanosKFStrikeConfig,
    gate_eval_table,
    run_sanos_kf_strike,
)
from quantdata.massive.client import MassiveClient

rows = []  # one metric dict per model, fed to um.comparison_table at the end

/Users/elie/Documents/GitHub/quant_data/quantdata/__init__.py:212: UserWarning: Using cache mode 'on' as specified in environment variable QUANTDATA_CACHE_MODE.


## CONFIG

In [2]:
TRADE_DATE = date(2024, 1, 3)
client = MassiveClient()
CALLS_ONLY, PUTS_ONLY = False, True
EVAL_IS_CALL = CALLS_ONLY if (CALLS_ONLY or PUTS_ONLY) else None

In [3]:
df_tickers = pl.read_csv(f"spy_options_tickers_{TRADE_DATE}.csv")

df_spy = client.fetch_trades("SPY", TRADE_DATE)

In [4]:
df_quotes, df_trades = build_iv_observations(df_tickers, TRADE_DATE, df_spy, min_trades=100)

Skipping O:SPY240105C00481000: 46 bad quotes
Skipping O:SPY240105C00482000: 64 bad quotes
Skipping O:SPY240105C00483000: 842 bad quotes
Skipping O:SPY240105C00484000: 2129 bad quotes
Skipping O:SPY240105C00485000: 3888 bad quotes
Skipping O:SPY240105P00462500: 17 bad quotes
Expiry 2024-01-05: implied forward from 13 strike(s)


In [5]:
import personal.Kalman_filter_spline.surface_model as sm

# df_quotes_iv / df_trades_iv are the IV-annotated outputs of build_iv_observations
# (already computed above as df_quotes / df_trades — just alias them here)
df_quotes_iv = df_quotes
df_trades_iv = df_trades
if EVAL_IS_CALL is not None:  # fit + eval on the same subset as the SANOS models (puts only)
    df_quotes_iv = df_quotes_iv.filter(pl.col("is_call") == EVAL_IS_CALL)
    df_trades_iv = df_trades_iv.filter(pl.col("is_call") == EVAL_IS_CALL)

In [6]:
grid_Q = [
    1e-8,
    1e-7,
    1e-6,
    1e-5,
    1e-4,
]  # process noise PER SECOND (batched fit ⇒ P_pred = P + Q·Δt)
grid_P0 = [
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
]  # initial uncertainty (large → flat start, KF converges quickly)
Q_structures = ["diagonal", "scaled_identity"]  # per-param scaling via model.q_diag_weights

# Spline surface KF config
# SVI_SLICE_MODEL: 5-param Gatheral SVI, single-expiry (good for short-dated SPY)
# Swap for sm.QUAD_SLICE_MODEL (3 params) or sm.BSPLINE_SLICE_MODEL (8 params) to compare
cfg = sm.SurfaceModelConfig(
    model=sm.SVI_SLICE_MODEL,
    Q_scale=1e-4,  # process noise PER SECOND (batched fit ⇒ P_pred = P + Q·Δt); retune vs per-tick
    Q_structure="diagonal",  # per-param scaling via model.q_diag_weights
    P0_scale=1e-3,  # initial uncertainty (large → flat start, KF converges quickly)
)

In [7]:
from personal.Kalman_filter_spline.surface_kalman_filters import run_spline_surface_kalman

for Q in grid_Q:
    for P0 in grid_P0:
        for Q_structure in Q_structures:
            cfg.Q_structure = Q_structure
            cfg.Q_scale = Q
            cfg.P0_scale = P0

            BATCH_S = (
                1.0  # fit & evaluate in 1-second batches (same methodology as the SANOS models)
            )

            q_res, t_res, df_state = run_spline_surface_kalman(
                df_quotes_iv, df_trades_iv, cfg, batch_seconds=BATCH_S
            )
            q = q_res.sort(["strike", "is_call", "ts_ns"]).to_pandas()

            # 1 s window id; within a window the spline shares one posterior θ and one held-out (previous-
            # window) prior θ, so prices vary only through each quote's own forward/T/k. Take the LAST quote
            # of each (strike, is_call) in the window as the per-second snapshot representative, then group by
            # the window — one cross-sectional snapshot per 1 s bucket, exactly like the SANOS 2-stage table.
            q["window"] = q["ts_ns"].astype("int64") // int(BATCH_S * 1e9)
            # snap = q.sort_values("ts_ns").drop_duplicates(subset=["window", "strike", "is_call"], keep="last")
            # held-out prior is the batched filter's own pre-update prediction; drop any row where it is
            # undefined (e.g. invalid forward/T at the snapshot quote).
            snap = q
            snap = snap.dropna(subset=["kalman_price_prior"])

            df_spline = um.from_option_prices(
                ts=snap[
                    "window"
                ],  # → n_snapshots = number of 1 s buckets (~24k), cross-sectional over strikes
                strike=snap["strike"],
                forward=snap["forward"],
                is_call=snap["is_call"],
                bid=snap["bid_price"],
                ask=snap["ask_price"],
                prior_price=snap["kalman_price_prior"],
                post_price=snap["kalman_price"],
                discount_factor=snap["discount_factor"] if "discount_factor" in snap else None,
            )
            # ── Negative mass: reconstruct smile over a strike grid at sampled snapshot times ──
            state_pd = df_state.to_pandas().sort_values("ts_ns").reset_index(drop=True)
            n_params = cfg.n_params
            theta_cols = [f"theta_{j}" for j in range(n_params)]

            # Strike grid spanning the observed strikes
            K_lo = float(df_quotes_iv["strike"].min())
            K_hi = float(df_quotes_iv["strike"].max())
            grid_K = np.linspace(K_lo, K_hi, 41)

            # Subsample snapshot times for speed (1 000 is usually plenty)
            n_snap = min(1000, len(state_pd))
            samp_idx = np.linspace(0, len(state_pd) - 1, n_snap, dtype=int)
            state_samp = state_pd.iloc[samp_idx].reset_index(drop=True)

            # For each sampled ts: look up the most-recent forward/T from the quote stream
            q_sorted = q.sort_values("ts_ns").reset_index(drop=True)
            asof = (
                np.searchsorted(q_sorted["ts_ns"].values, state_samp["ts_ns"].values, side="right")
                - 1
            )
            asof = np.clip(asof, 0, len(q_sorted) - 1)
            F_vec = q_sorted["forward"].values[asof]
            T_vec = q_sorted["T"].values[asof]
            thetas_samp = state_samp[theta_cols].values  # (n_snap, n_params)

            # Reconstruct call-price matrix (n_snap, n_grid)
            C_mat = np.full((n_snap, len(grid_K)), np.nan)
            for i in range(n_snap):
                theta_i, F_i, T_i = thetas_samp[i], F_vec[i], T_vec[i]
                if not (np.isfinite(theta_i).all() and F_i > 0 and np.isfinite(F_i) and T_i > 0):
                    continue
                ivs = np.array(
                    [cfg.model.observe(theta_i, np.log(k / F_i), T_i)[0] for k in grid_K]
                )
                C_mat[i] = um.bs_call_price(
                    np.full(len(grid_K), F_i), grid_K, np.full(len(grid_K), T_i), ivs
                )

            valid = np.isfinite(C_mat).all(axis=1)
            nm = um.negative_mass_surface(grid_K, C_mat[valid], F_vec[valid])

            m_spline = um.evaluate(
                df_spline,
                model_name=f"KF spline-{cfg.Q_scale}-{cfg.P0_scale}-{cfg.Q_structure}",
                neg_mass_values=nm,
            )
            rows.append(m_spline)
            print(m_spline)

{'model': 'KF spline-1e-08-1e-06-diagonal', 'ho_nse_mean': 0.8019186254207415, 'ho_nse_med': 0.7617232781907399, 'ho_nse_std': 0.7287770334425812, 'is_nse_mean': 0.6894087298657382, 'is_nse_med': 0.6833151037601376, 'is_nse_std': 0.3170227494473547, 'ho_rmse_mean': 0.03199633258484943, 'ho_rmse_med': 0.03146970566396938, 'ho_rmse_std': 0.023469265216631032, 'is_rmse_mean': 0.03136827343352806, 'is_rmse_med': 0.031183261417802692, 'is_rmse_std': 0.02301968569812365, 'neg_mass_mean': -0.0005283085504451541, 'neg_mass_med': 0.0, 'arb_frac': 0.212, 'n_snapshots': 24066}
{'model': 'KF spline-1e-08-1e-06-scaled_identity', 'ho_nse_mean': 0.7757304223785529, 'ho_nse_med': 0.738402684761746, 'ho_nse_std': 0.7350814713738685, 'is_nse_mean': 0.6494386550657193, 'is_nse_med': 0.6443043849490224, 'is_nse_std': 0.3237543488099142, 'ho_rmse_mean': 0.03242216991819608, 'ho_rmse_med': 0.03201104113611286, 'ho_rmse_std': 0.02383103689891435, 'is_rmse_mean': 0.0316823347687359, 'is_rmse_med': 0.031725838

In [8]:
df = um.comparison_table(rows)

In [12]:
df.sort_values("ho_nse_med").to_csv(f"spy_options_kf_spline_eval_{TRADE_DATE}.csv", index=True)

In [10]:
rows

[{'model': 'KF spline-1e-08-1e-06-diagonal',
  'ho_nse_mean': 0.8019186254207415,
  'ho_nse_med': 0.7617232781907399,
  'ho_nse_std': 0.7287770334425812,
  'is_nse_mean': 0.6894087298657382,
  'is_nse_med': 0.6833151037601376,
  'is_nse_std': 0.3170227494473547,
  'ho_rmse_mean': 0.03199633258484943,
  'ho_rmse_med': 0.03146970566396938,
  'ho_rmse_std': 0.023469265216631032,
  'is_rmse_mean': 0.03136827343352806,
  'is_rmse_med': 0.031183261417802692,
  'is_rmse_std': 0.02301968569812365,
  'neg_mass_mean': -0.0005283085504451541,
  'neg_mass_med': 0.0,
  'arb_frac': 0.212,
  'n_snapshots': 24066},
 {'model': 'KF spline-1e-08-1e-06-scaled_identity',
  'ho_nse_mean': 0.7757304223785529,
  'ho_nse_med': 0.738402684761746,
  'ho_nse_std': 0.7350814713738685,
  'is_nse_mean': 0.6494386550657193,
  'is_nse_med': 0.6443043849490224,
  'is_nse_std': 0.3237543488099142,
  'ho_rmse_mean': 0.03242216991819608,
  'ho_rmse_med': 0.03201104113611286,
  'ho_rmse_std': 0.02383103689891435,
  'is_rms